### Getting the deputies data

We will work with the LVI (56th) period of the "Cámara de Diputadas y Diputados" (Deputies' Chamber), the dates of this period are from 2022-03-11 to 2026-03-10.

We want to get the deputy ID (dipid), names, gender, date of birth and political militancies.

Since 2026-03-11 starts a new cohort of deputies I will use a specific call to get the information of the deputies, instead of actual period, this will not work while working the code and also not past the 11-03

There's a difference between Periodo legislativo (4 year term of the Chamber), and the Legislatura (1year subdivision in the periodo Legislativo)

In [9]:
import requests
import sqlite3
import pandas as pd
import xml.etree.ElementTree as et
from datetime import datetime


From the database that the Biblioteca Nacional del Congreso (ADD THE LINK) we see that the Periodo legislativo that we want to work with is the officially named LVI (56th), but in the inner database the whole period is numbered with ID=10

[This is where we can find the endpoints from the Cámara del Congreso](https://www.camara.cl/transparencia/datosAbiertos.aspx)

In [10]:
# this is the endpoint URL that we will work on
url = "https://opendata.camara.cl/camaradiputados/WServices/WSDiputado.asmx/retornarDiputadosXPeriodo"

# calling the API
# the id we are usign now is 10, that's the Period 2022-2026
response = requests.get(url, params={"prmPeriodoID":10}) #add timeout?

# this is the answer we get
deputies_xml = response.text

# save the data so we don't have to download it again
with open(f"diputados_periodo_10.xml", "w", encoding="utf-8") as f:
    f.write(deputies_xml)

In [11]:
deputies_xml

'<?xml version="1.0" encoding="utf-8"?>\r\n<DiputadosPeriodoColeccion xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns="http://opendata.camara.cl/camaradiputados/v1">\r\n  <DiputadoPeriodo>\r\n    <FechaInicio>2026-03-10T23:59:59</FechaInicio>\r\n    <FechaTermino xsi:nil="true" />\r\n    <Diputado>\r\n      <Id>1096</Id>\r\n      <Nombre>María Candelaria</Nombre>\r\n      <Nombre2 />\r\n      <ApellidoPaterno>Acevedo</ApellidoPaterno>\r\n      <ApellidoMaterno>Sáez</ApellidoMaterno>\r\n      <FechaNacimiento>1958-09-12T00:00:00</FechaNacimiento>\r\n      <RUT />\r\n      <RUTDV />\r\n      <Sexo Valor="0">Femenino</Sexo>\r\n      <Militancias>\r\n        <Militancia>\r\n          <FechaInicio>2022-03-11T00:00:00</FechaInicio>\r\n          <FechaTermino>2026-03-10T23:59:59</FechaTermino>\r\n          <Partido>\r\n            <Id>PC</Id>\r\n            <Nombre>Partido Comunista</Nombre>\r\n            <Alias>PC</Alias>\r\n         

In [12]:
# parse XML using ElementTree, setting the data in a tree structure
deputies_parsed = et.fromstring(deputies_xml)
deputies_parsed

# since in the tree structure the namespace gets inhered fir every children and grandchildren of the root, we have to clean and reduce it
# if we want to work with the tags we need to find a way to work also with the namespace
ns = {"ns": "http://opendata.camara.cl/camaradiputados/v1"}





In [ ]:
# we create list to store the inform
deputies_list = []
parties_list = []
militancies_list = []

# loop over each deputy
for dp in deputies_parsed.findall("ns:DiputadoPeriodo", ns):
    # first the information on the term boundaries
    term_start = dp.findtext("ns:FechaInicio", default=None, namespaces=ns)
    term_end = dp.findtext("ns:FechaTermino", default=None, namespaces=ns)
    # now we know that the term_start is really the end of the electoral process

    # now nested inside there's the concrete information on the deputy
    diputado = dp.find("ns:Diputado", ns)
    if diputado is None:
        continue

    diputado_id = diputado.findtext("ns:Id", default=None, namespaces=ns)
    if diputado_id is None:
        continue
    dipid = int(diputado_id)

    # the sex element has text and attribute
    sex_element = diputado.find("ns:Sexo", ns)
    sex_text = sex_element.text if sex_element is not None else None
    sex_value = sex_element.attrib.get("Valor") if sex_element is not None else None
    
    deputies_list.append({
        "dipid": dipid,
        "nombre": clean(diputado.findtext("ns:Nombre", default=None, namespaces=ns)), # some deputies have two names, but they are all in the same tag
        "apellido_paterno": clean(diputado.findtext("ns:ApellidoPaterno", default=None, namespaces=ns)),
        "apellido_materno": clean(diputado.findtext("ns:ApellidoMaterno", default=None, namespaces=ns)),
        "fecha_nacimiento": clean(diputado.findtext("ns:FechaNacimiento", default=None, namespaces=ns)),
        "sex_text": sex_text,
        "sex_value": sex_value
    })

    # now we proceeed with the information on the militancies
    militancias = diputado.find("ns:Militancias", ns)
    if militancias is None:
        continue
    for i in militancias.findall("ns:Militancia", ns):
        i_start = i.findtext("ns:FechaInicio", default=None, namespaces=ns)
        i_end = i.findtext("ns:FechaTermino", default=None, namespaces=ns)

        partido = i.find("ns:Partido", ns)
        if partido is None:
            continue

        party_id = partido.findtext("ns:Id", default=None, namespaces=ns)
        party_name = partido.findtext("ns:Nombre", default=None, namespaces=ns)
        party_alias = partido.findtext("ns:Alias", default=None, namespaces=ns)

        # we save the information on the parties that conform the Congress
        parties_list.append({
            "party_id": party_id,
            "party_name": party_name,
            "party_alias": party_alias
        })

        # finally we save a table linking deputies to their parties
        militancies_list.append({
            "dipid": dipid,
            "party_id": party_id,
            "date_start": i_start,
            "date_end": i_end
        })
    
    dip_df = pd.DataFrame(deputies_list)
    parties_df = pd.DataFrame(parties_list).drop_duplicates(subset=["party_id"])
    militancies_df = pd.DataFrame(militancies_list).drop_duplicates(subset=["dipid", "party_id", "date_start"])

For now we aren't trying to get the deputies making changes, if I want to do it later I have to consider distinguishing between the instances where someone is registerd as changing party and changes to the same one PS -> PS -> PS (e.g. Jaime Naranjo)

## SQL tables

now we try to get the lists into the SQL tables

In [23]:
con = sqlite3.connect("chilean_polispeech.db")
cur = con.cursor()
cur.execute("""
CREATE TABLE IF NOT EXISTS deputies (
    dipid INT PRIMARY KEY, 
    name TEXT,
    lastname_1 TEXT,
    lastname_2 TEXT,
    birth_date TEXT,            
    sex_value TEXT,            
    sex_text TEXT            
);

CREATE TABLE IF NOT EXISTS parties (
    party_id INT PRIMARY KEY,        
    party_name TEXT,
    party_alias TEXT
);

CREATE TABLE IF NOT EXISTS militancies (
    dipid INT,
    party_id INT,
    date_start TEXT,
    date_end TEXT,
    PRIMARY KEY (dipid, party_id, date_start),
    FOREIGN KEY (dipid) REFERENCES deputies (dipid),
    FOREIGN KEY (party_id) REFERENCES parties (party_id)
);
""")
con.commit()

dip_df.to_sql("deputies", con, if_exists="append", index=False)
parties_df.to_sql("parties", con, if_exists="append", index=False)
militancies_df.to_sql("militancies", con, if_exists="append", index=False)
con.close()


ProgrammingError: You can only execute one statement at a time.

In [ ]:
type(militancies_list)

In [ ]:
deputies_parsed.get(".//ns:diputado", ns)